## User story
Our org is retiring an older web map that appears in many ArcGIS StoryMaps. We want to update stories so any map block that references the **old map item** is replaced with the **new map item**.

## What this notebook does
1. Signs in to your portal/org.
2. Opens each StoryMap in a provided list (or search results).
3. Scans the story’s content blocks for `Map` blocks that reference the old map item ID.
4. Replaces those blocks to reference the new map item (including layer info).
5. Saves the story.

## Before you run it
- Create a `.env` file (or set env vars) with `USERNAME_TARGET` and `PASSWORD_TARGET`.
- Decide whether you’re updating a few known StoryMaps (recommended first) or doing a broader search.
- Start with a small list of IDs to validate results before scaling up.

In [ ]:
import arcgis
from arcgis.gis import GIS, Item
from arcgis.apps.storymap import StoryMap, Map
from arcgis.apps.storymap.story_content import Text, Embed
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings("ignore")

In [ ]:
EXPECTED_ARCGIS_VERSION = "2.4.3"
print("ArcGIS API for Python version:", arcgis.__version__)
if arcgis.__version__ != EXPECTED_ARCGIS_VERSION:
    print(
        "Note: this notebook was authored against",
        EXPECTED_ARCGIS_VERSION,
        "(you have",
        arcgis.__version__ + ")",
    )

In [ ]:
# Load environment variables from .env (or your OS environment)
load_dotenv()

USERNAME_TARGET = os.getenv("USERNAME_TARGET")
PASSWORD_TARGET = os.getenv("PASSWORD_TARGET")

if not USERNAME_TARGET or not PASSWORD_TARGET:
    raise ValueError(
        "Missing credentials. Set USERNAME_TARGET and PASSWORD_TARGET in a .env file (or env vars)."
    )

In [ ]:
PORTAL_URL = "https://www.arcgis.com"  # For Enterprise, set to your portal URL
gis_con = GIS(PORTAL_URL, USERNAME_TARGET, PASSWORD_TARGET)

print("Signed in as:", gis_con.users.me.username)

## Configuration
You’ll set three things:
- **Stories to update**: a list of StoryMap item IDs (start small).
- **Old map item ID**: the web map you’re retiring.
- **New map item ID**: the web map that should replace it.

Tip: To build a large list of stories, you can start from a search like:
```python
stories = gis_con.content.advanced_search(query="type:StoryMap", max_items=1000)
story_ids = [item.id for item in stories["results"]]
```

In [ ]:
# Start with a small test set of StoryMap IDs
storymap_ids_to_update = [""] # Add StoryMap item IDs you want to update here

OLD_WEBMAP_ITEM_ID = "" # Add the webmap item ID that you want to replace in the stories (can be found in the URL of the webmap, or in the item details page)
NEW_WEBMAP_ITEM_ID = "" # Add the webmap item ID that you want to use as replacement (can be found in the URL of the webmap, or in the item details page)

# Safety switch: set to True only after validating results on a test story
SAVE_CHANGES = False

## Replace map blocks
For each StoryMap, this loop:
- Loads the story content blocks
- Finds `Map` blocks pointing at `OLD_WEBMAP_ITEM_ID`
- Replaces them with `NEW_WEBMAP_ITEM_ID`
- Saves the story (only if `SAVE_CHANGES = True`)

In [ ]:
new_map_item = Item(gis_con, NEW_WEBMAP_ITEM_ID)
new_map_instance = Map(new_map_item)
new_map_layers = new_map_instance.map_layers

total_updated_blocks = 0
total_stories_with_updates = 0

for story_id in storymap_ids_to_update:
    print(f"\nStoryMap: {story_id}")
    try:
        story = StoryMap(story_id, gis_con)
        updated_in_this_story = 0

        for block in story.contents:
            if not isinstance(block, Map):
                continue
            if getattr(block.map, "itemid", None) != OLD_WEBMAP_ITEM_ID:
                continue

            updated_in_this_story += 1
            # Swap the map reference
            block.map = new_map_instance
            # Update map layers in the underlying resource dict to match the new map
            block.properties["resource_dict"]["data"]["mapLayers"] = new_map_layers

        if updated_in_this_story:
            total_updated_blocks += updated_in_this_story
            total_stories_with_updates += 1
            print(f"  Updated map blocks: {updated_in_this_story}")
            if SAVE_CHANGES:
                story.save()
                print("  Saved story")
            else:
                print("  Dry run (not saved). Set SAVE_CHANGES = True to persist.")
        else:
            print("  No matching map blocks found")
    except Exception as e:
        print(f"  Error processing {story_id}: {e}")

print(f"\nDone. Stories updated: {total_stories_with_updates}, blocks updated: {total_updated_blocks}")

## The bigger pattern
The same approach works for other block types where you can uniquely identify what to change (for example: embeds, text links, or logos).
The snippets below are *examples* of what you could do inside the same `for block in story.contents` loop.

In [ ]:
# Embeds
# ... in content loop
if isinstance(content_item, Embed) and content_item.link=="RETIRING_LINK":
    content_item.link = "REPLACEMENT_LINK"
#... save storymap after loop


In [ ]:
# Logo
# no need for loop, use storymap directly
this_storymap.set_logo(link="URL")
# or from path
this_storymap.set_logo(image="PATH_TO_IMAGE")

In [ ]:
# Text links
# ... in content loop
if isinstance(content_item, Text) and "RETIRING_LINK" in content_item.text:
    content_item.text = content_item.text.replace("RETIRING_LINK", "REPLACEMENT_LINK")
#... save storymap after loop
